# 09 • Priorisation sous capacité et dérive

`[MÉTA | Formation 4-024 | Niveau Application | TP 09 | Mode CPU local]`

**Objectif :** Relier le classement à une capacité de revue humaine.

**Temps indicatif :** Atelier J3 ou extension de TP 03. Ces temps sont répartis dans le conducteur, pas additionnés hors des 18 heures.

**Prérequis :** Métriques de classification.

**Preuves de réussite :** Top-k mesuré, prévalence indiquée, limite des labels explicitée.

**Sources :** R16, R17, R19 ; scénario entièrement fictif.

Les jeux métier sont synthétiques. Aucun fichier personnel ou fiscal réel ne doit être chargé. Les résultats obtenus ici ne constituent pas une validation métier.

**Mode d’emploi :** exécuter les cellules dans l’ordre. Les cellules d’exercice du cahier apprenant sont à compléter ; le corrigé contient le code et des résultats de référence sur CPU.

In [1]:
from pathlib import Path
import sys, os, json
# Chercher le kit depuis le répertoire du notebook ou celui de lancement.
HERE = Path.cwd().resolve()
TP_ROOT = next((p for p in [HERE, *HERE.parents] if (p / "modules" / "atelier.py").exists()), None)
if TP_ROOT is None:
    raise FileNotFoundError("Ouvrir ce notebook depuis le dossier 03_Travaux_pratiques du kit décompressé.")
sys.path.insert(0, str(TP_ROOT / "modules"))
os.environ.setdefault("KERAS_BACKEND", "torch")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch
from torch import nn
from atelier import *
seed_all(42)
print("Moteur disponible :", torch.__version__, "| Données :", DATA)


Moteur disponible : 2.10.0+cpu | Données : /mnt/data/deep_learning_4_024/03_Travaux_pratiques/donnees


## 1. Données rares et protocole
5 000 observations synthétiques, environ 2,5 % de positifs. Les proportions exactes sont affichées. La cible représente l’utilité d’une revue, pas une culpabilité. On compare un modèle linéaire, des arbres et un petit MLP.

In [2]:
d=split_tabular(True);tr=d['train'];X,y=d['validation']
print('Prévalence train / validation :',tr[1].mean(),y.mean())
models={'logistique':LogisticRegression(max_iter=300).fit(*tr),'arbres':HistGradientBoostingClassifier(max_iter=80,random_state=42).fit(*tr)}
seed_all();mlp=BinaryMLP();h=fit_model(mlp,tr,(X,y),epochs=16)
scores={name:model.predict_proba(X)[:,1] for name,model in models.items()};scores['MLP']=predict(mlp,X)
rows=[]
for name,s in scores.items():rows.append({'modele':name,**binary_metrics(y,s),**topk_metrics(y,s,30)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

Prévalence train / validation : 0.026285714285714287 0.02666666666666667


    modele  seuil  exactitude  precision  rappel     f1  roc_auc  precision_moyenne_AP  brier  alertes  k  cas_pertinents  precision_a_k  rappel_a_k
logistique    0.5      0.9827     1.0000    0.35 0.5185   0.8900                0.6365 0.0150        7 30              12         0.4000         0.6
    arbres    0.5      0.9880     1.0000    0.55 0.7097   0.9544                0.8378 0.0112       11 30              16         0.5333         0.8
       MLP    0.5      0.9867     0.9167    0.55 0.6875   0.9415                0.8050 0.0094       12 30              16         0.5333         0.8


## 2. Choisir pour 30 revues
Choisir selon le nombre de cas utiles dans les 30 premiers, puis la précision moyenne en cas d’égalité. Ce critère est une hypothèse métier fictive. Le modèle sélectionné n’est pas déclaré meilleur pour toutes les capacités ni pour les données futures.

In [3]:
choisi=max(rows,key=lambda r:(r['cas_pertinents'],r['precision_moyenne_AP']))
print('Choix sur validation :',choisi['modele'],'| cas utiles dans 30 :',choisi['cas_pertinents'])
print('Nombre total de positifs validation :',int(y.sum()))
assert choisi['k']==30
save_result('09_capacite',{'validation':rows,'choix':choisi['modele'],'test_ouvert':False})

Choix sur validation : arbres | cas utiles dans 30 : 16
Nombre total de positifs validation : 20


PosixPath('/mnt/data/deep_learning_4_024/03_Travaux_pratiques/resultats/09_capacite.json')

## 3. Simuler un changement de distribution
Décaler deux variables ne reproduit pas une stratégie réelle de fraude. Ce test explore une fragilité numérique. Le label est conservé artificiellement ; on ne peut donc pas en déduire un mécanisme causal de dérive réelle.

In [4]:
Xd=X.copy();Xd[:,0]+=2.;Xd[:,1]*=1.8
avant=scores['MLP'];apres=predict(mlp,Xd)
print('Score moyen initial / perturbé :',avant.mean(),apres.mean())
fig,ax=plt.subplots(figsize=(7,4));ax.hist(avant,bins=20,alpha=.5,label='Initial');ax.hist(apres,bins=20,alpha=.5,label='Perturbé')
ax.set(xlabel='Score',ylabel='Effectif',title='Simulation contrôlée, non preuve métier');ax.legend();fig.tight_layout();fig.savefig(RESULTS/'09_derive.png',dpi=150)

Score moyen initial / perturbé : 0.022492867 0.017019251


## 4. Labels retardés et angle mort
Les enquêteurs ne labellisent pas toutes les transactions. Les performances mesurées uniquement parmi les alertes peuvent ignorer les cas jamais examinés. Décrire un protocole d’audit avec échantillonnage de contrôle, délai de retour des labels et supervision effective. Ce raisonnement vient du cas de fraude en flux [R19], pas d’un détail technique publié de CFVR.